## Feature Ablation Studies

Ablate features and determine their downstream effects.

Threshold finding is done on the **train set only**; all filter evaluations use the held-out **test set**.


#### 1. Load data & models

In [1]:
# imports

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, auc as sk_auc
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import ROOT_DIR, OUTPUT_DIR, EXCLUSION_LIST, LATENT_DIM
from src.data_processor import process_all_cases, load_processed_data
from src.model import TopKSAE

OUTPUT_DIR = '/Users/bridget/Desktop/projects/laloo-sae/processed_data'
MODEL_DIR  = os.path.expanduser('~/Desktop/projects/laloo-sae/models/12_21_25')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cpu


In [2]:
# load data
latents_normalized, metadata, stats = load_processed_data(OUTPUT_DIR)

print(f"Latent shape:   {latents_normalized.shape}")
print(f"Metadata shape: {metadata.shape}")
print(metadata.head(2))


Loaded 435,069 samples from /Users/bridget/Desktop/projects/laloo-sae/processed_data
Latent shape:   (435069, 30)
Metadata shape: (435069, 6)
               case_id  sample_idx      rmsd        energy  generation  \
0  afab_2nnq_afab_3fr4           0  2.385954  -5425.862544          15   
1  afab_2nnq_afab_3fr4           1  3.430689  10000.000000          15   

   global_idx  
0           0  
1           1  


In [3]:
# load train/test splits (case-stratified, from splits.npz)
splits    = np.load(os.path.join(OUTPUT_DIR, 'splits.npz'), allow_pickle=True)
train_idx = splits['train_idx']
test_idx  = splits['test_idx']

latents_train  = latents_normalized[train_idx]
latents_test   = latents_normalized[test_idx]
metadata_train = metadata.iloc[train_idx].reset_index(drop=True)
metadata_test  = metadata.iloc[test_idx].reset_index(drop=True)

RMSD_THRESH = 2.0

print(f"Train: {len(train_idx):,}  |  Test: {len(test_idx):,}")
print(f"Train bad-pose rate: {(metadata_train['rmsd'] > RMSD_THRESH).mean():.3f}")
print(f"Test  bad-pose rate: {(metadata_test['rmsd']  > RMSD_THRESH).mean():.3f}")


Train: 294,851  |  Test: 68,614
Train bad-pose rate: 0.874
Test  bad-pose rate: 0.853


In [4]:
# load models
k_values = {1, 3, 6, 8, 15, 20, 30}
trained_models = {}
results_summary = torch.load(os.path.join(MODEL_DIR, 'training_summary.pkl'))

for k in k_values:
    if k in results_summary and len(results_summary[k]) > 0:
        best_run = min(results_summary[k], key=lambda r: r['best_val_loss'])
        run_id   = best_run['run_id']
        model_path = os.path.join(MODEL_DIR, f'topksae_k{k}_run{run_id}.pt')
        if os.path.exists(model_path):
            model = TopKSAE(input_dim=LATENT_DIM, hidden_dim=120, k=k)
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.to(device)
            model.eval()
            trained_models[k] = model
            print(f"Loaded k={k}, run {run_id}, val_loss={best_run['best_val_loss']:.6f}")
        else:
            print(f"k={k}: model file not found at {model_path}")
    else:
        print(f"k={k}: no training results found")


Loaded k=1, run 2, val_loss=0.524500
Loaded k=3, run 4, val_loss=0.387856
Loaded k=20, run 0, val_loss=0.013269
Loaded k=6, run 0, val_loss=0.273244
Loaded k=8, run 4, val_loss=0.221989
Loaded k=30, run 3, val_loss=0.000093
Loaded k=15, run 2, val_loss=0.051162


### 2. Feature Discovery/Loading

Top activating poses for k=3 features 39, 111, 41 (from prior correlation analysis).


In [6]:
def get_top_cases(feature_idx, activations, meta, n=20):
    top_indices = torch.argsort(activations[:, feature_idx], descending=True)[:n]
    return meta.iloc[top_indices.cpu().numpy()]


In [7]:
# k=3 activations — extract separately for train and test
model = trained_models[3]
model.eval()

with torch.no_grad():
    f_acts_train = model.get_acts(
        torch.as_tensor(latents_train, dtype=torch.float32, device=device)
    ).cpu()
    f_acts_test = model.get_acts(
        torch.as_tensor(latents_test,  dtype=torch.float32, device=device)
    ).cpu()

print(f"Train activations: {f_acts_train.shape}")
print(f"Test  activations: {f_acts_test.shape}")

# Top cases from test set (for display only)
for feat in [39, 111, 41]:
    top = get_top_cases(feat, f_acts_test, metadata_test, n=5)
    print(f"\nTop 5 test poses — feature {feat}:")
    print(top[['case_id', 'rmsd', 'energy']].to_string(index=False))


Train activations: torch.Size([294851, 120])
Test  activations: torch.Size([68614, 120])

Top 5 test poses — feature 39:
              case_id     rmsd  energy
throm_1ae8_throm_1d9i 5.792728 10000.0
throm_1ae8_throm_1d9i 6.313361 10000.0
throm_1ae8_throm_1d9i 5.611657 10000.0
throm_1ae8_throm_1d9i 8.773404 10000.0
throm_1ae8_throm_1d9i 7.352363 10000.0

Top 5 test poses — feature 111:
              case_id     rmsd  energy
throm_1ae8_throm_1d9i 5.218159 10000.0
throm_1ae8_throm_1d9i 5.510282 10000.0
throm_1ae8_throm_1d9i 5.678670 10000.0
throm_1ae8_throm_1d9i 3.855839 10000.0
throm_1ae8_throm_1d9i 4.638728 10000.0

Top 5 test poses — feature 41:
              case_id     rmsd        energy
glur5_3fv1_glur5_3fuz 1.720595  10000.000000
glur5_3fv1_glur5_3fuz 1.106975 -10576.489379
glur5_3fv1_glur5_3fuz 0.579706 -10510.586677
glur5_3fv1_glur5_3fuz 0.851268 -10584.350866
glur5_3fv1_glur5_3fuz 0.444705 -10571.664349


### 3. Feature Analysis

In [8]:
def analyze_feature_physics(feature_idx, activations, meta):
    feat_acts = activations[:, feature_idx].numpy()

    corr_rmsd   = pd.Series(feat_acts).corr(meta['rmsd'],   method='spearman')
    corr_energy = pd.Series(feat_acts).corr(meta['energy'], method='spearman')

    print(f"--- Feature {feature_idx} (train set) ---")
    print(f"  Spearman ρ with RMSD:   {corr_rmsd:.3f}")
    print(f"  Spearman ρ with Energy: {corr_energy:.3f}")

    high_act_mask   = feat_acts > (feat_acts.mean() + 2*feat_acts.std())
    avg_rmsd_high   = meta[high_act_mask]['rmsd'].mean()
    print(f"  Mean RMSD when feature fires (>mu+2σ): {avg_rmsd_high:.2f}Å")

for f in [39, 111, 41]:
    analyze_feature_physics(f, f_acts_train, metadata_train)


--- Feature 39 (train set) ---
  Spearman ρ with RMSD:   0.090
  Spearman ρ with Energy: 0.133
  Mean RMSD when feature fires (>mu+2σ): 7.56Å
--- Feature 111 (train set) ---
  Spearman ρ with RMSD:   0.016
  Spearman ρ with Energy: -0.040
  Mean RMSD when feature fires (>mu+2σ): 5.58Å
--- Feature 41 (train set) ---
  Spearman ρ with RMSD:   0.089
  Spearman ρ with Energy: -0.012
  Mean RMSD when feature fires (>mu+2σ): 6.53Å


In [9]:
# SAE representation quality vs raw latents — train/test from splits.npz
y_train = (metadata_train['rmsd'] > RMSD_THRESH).astype(int).values
y_test  = (metadata_test['rmsd']  > RMSD_THRESH).astype(int).values

X_sae_train = f_acts_train.numpy()
X_sae_test  = f_acts_test.numpy()
X_raw_train = latents_train
X_raw_test  = latents_test

print("Classifier: LogisticRegression — train on train_idx, evaluate on test_idx\n")
for name, X_tr, X_te in [
    ('SAE k=3 (all features)', X_sae_train, X_sae_test),
    ('Raw latents',             X_raw_train, X_raw_test),
]:
    clf = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced').fit(X_tr, y_train)
    probs = clf.predict_proba(X_te)[:, 1]
    auc  = roc_auc_score(y_test, probs)
    aupr = average_precision_score(y_test, probs)
    print(f"{name:<35}  ROC-AUC={auc:.4f}  auPR={aupr:.4f}")


Classifier: LogisticRegression — train on train_idx, evaluate on test_idx

SAE k=3 (all features)               ROC-AUC=0.5102  auPR=0.8560
Raw latents                          ROC-AUC=0.5127  auPR=0.8554


### 4. Feature scoring across k=8 and k=15 models

For each feature: precision lift (P(good pose | fires) / baseline) and within-case discrimination.
Scored on the **train set** only.


In [10]:
RMSD_THRESH = 2.0
y_train_good = (metadata_train['rmsd'] <= RMSD_THRESH).astype(int).values
baseline_rate = y_train_good.mean()
print(f"Baseline good-pose rate (train): {baseline_rate:.3f}  ({baseline_rate*100:.1f}%)")

def score_features(model, meta_tr, latents_tr):
    model.eval()
    with torch.no_grad():
        acts = model.get_acts(
            torch.as_tensor(latents_tr, dtype=torch.float32, device=device)
        ).cpu().numpy()

    n_feat    = acts.shape[1]
    y_good    = (meta_tr['rmsd'] <= RMSD_THRESH).astype(int).values
    records   = []
    meta_base = meta_tr[['case_id']].copy()
    meta_base['is_good'] = y_good

    for f in range(n_feat):
        a        = acts[:, f]
        active   = a > 0
        n_active = active.sum()
        if n_active < 50:
            continue
        sparsity  = n_active / len(a)
        precision = y_good[active].mean()
        lift      = precision / baseline_rate

        meta_f = meta_base.copy()
        meta_f['act'] = a
        grp = meta_f.groupby(['case_id', 'is_good'])['act'].mean().unstack('is_good')
        grp.columns.name = None
        within_disc = (grp[1] - grp[0]).dropna().mean() if (0 in grp.columns and 1 in grp.columns) else float('nan')

        records.append({'feature': f, 'sparsity': sparsity, 'n_active': n_active,
                        'precision': precision, 'lift': lift, 'within_disc': within_disc})

    return pd.DataFrame(records).sort_values('lift', ascending=False).reset_index(drop=True), acts

feat_results   = {}
acts_train_byk = {}
acts_test_byk  = {}
for k in [8, 15]:
    if k not in trained_models:
        print(f"k={k} not loaded, skipping"); continue
    print(f"\nScoring k={k} features (train set)...")
    df, acts_tr = score_features(trained_models[k], metadata_train, latents_train)
    feat_results[k]   = df
    acts_train_byk[k] = acts_tr
    with torch.no_grad():
        acts_test_byk[k] = trained_models[k].get_acts(
            torch.as_tensor(latents_test, dtype=torch.float32, device=device)
        ).cpu().numpy()
    print(f"  {len(df)} features fire on ≥50 train samples")
    print(df.head(10)[['feature','sparsity','precision','lift','within_disc']].to_string(index=False))


Baseline good-pose rate (train): 0.126  (12.6%)

Scoring k=8 features (train set)...
  120 features fire on ≥50 train samples
 feature  sparsity  precision     lift  within_disc
     115  0.025657   0.431461 3.415028     0.153394
      88  0.043171   0.417786 3.306793     0.067210
     106  0.008309   0.368163 2.914026     0.140541
     113  0.032372   0.292404 2.314392     0.114312
      59  0.015031   0.260605 2.062696     0.110492
       6  0.103727   0.254578 2.014991     0.131407
      91  0.062815   0.250310 1.981217     0.012576
      93  0.017782   0.239176 1.893088     0.062402
      25  0.054434   0.234268 1.854239     0.043393
      80  0.060651   0.226360 1.791650    -0.087586

Scoring k=15 features (train set)...
  120 features fire on ≥50 train samples
 feature  sparsity  precision     lift  within_disc
      47  0.005406   0.891468 7.056003     0.012011
      15  0.002822   0.887019 7.020791     0.020164
     104  0.001872   0.833333 6.595865     0.007952
     110  0.001

In [11]:
y_test_good = (metadata_test['rmsd'] <= RMSD_THRESH).astype(int).values

for k, df in feat_results.items():
    acts_te = acts_test_byk[k]
    top5    = df.head(5)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'k={k} — top 5 good-pose features (scored on train, evaluated on test)', fontsize=12)
    colors = plt.cm.tab10.colors

    for i, (_, row) in enumerate(top5.iterrows()):
        f     = int(row['feature'])
        score = acts_te[:, f]
        fpr, tpr, _  = roc_curve(y_test_good, score)
        prec, rec, _ = precision_recall_curve(y_test_good, score)
        rauc  = sk_auc(fpr, tpr)
        prauc = sk_auc(rec, prec)

        label = f"feat {f}  lift={row['lift']:.2f}  AUC={rauc:.3f}"
        axes[0].plot(fpr, tpr,  color=colors[i], lw=1.5, label=label)
        axes[1].plot(rec, prec, color=colors[i], lw=1.5, label=f"feat {f}  PR-AUC={prauc:.3f}")

    axes[0].plot([0,1],[0,1],'k--',lw=1)
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR (good-pose recall)')
    axes[0].set_title('ROC — good pose detection (test set)')
    axes[0].legend(fontsize=7, loc='lower right')

    baseline_test = y_test_good.mean()
    axes[1].axhline(baseline_test, color='grey', linestyle=':', lw=1,
                    label=f'baseline {baseline_test:.2f}')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('PR — good pose detection (test set)')
    axes[1].legend(fontsize=7, loc='upper right')

    plt.tight_layout()
    plt.show()


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Feature quality: precision lift vs sparsity (train set)\n'
             'Good filter features: high lift + low sparsity (top-left)', fontsize=11)

for ax, (k, df) in zip(axes, feat_results.items()):
    sc = ax.scatter(df['sparsity'], df['lift'],
                    c=df['within_disc'], cmap='RdYlGn',
                    s=20, alpha=0.7, vmin=-0.01, vmax=0.05)
    plt.colorbar(sc, ax=ax, label='within-case disc.')
    ax.axhline(1.0, color='grey', linestyle='--', lw=1, label='lift = 1')
    for _, row in df.head(5).iterrows():
        ax.annotate(f"f{int(row['feature'])}",
                    (row['sparsity'], row['lift']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel('Sparsity (fraction of train samples active)')
    ax.set_ylabel('Precision lift over baseline')
    ax.set_title(f'k={k} ({len(df)} features)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

if len(feat_results) == 2:
    top_k8  = set(feat_results[8].head(10)['feature'].astype(int))
    top_k15 = set(feat_results[15].head(10)['feature'].astype(int))
    shared  = top_k8 & top_k15
    print(f"Top-10 in k=8:  {sorted(top_k8)}")
    print(f"Top-10 in k=15: {sorted(top_k15)}")
    print(f"Shared:         {sorted(shared)}")


Top-10 in k=8:  [6, 25, 59, 80, 88, 91, 93, 106, 113, 115]
Top-10 in k=15: [2, 15, 26, 28, 47, 74, 75, 95, 104, 110]
Shared:         []


### 5. Threshold-Based Bad-Pose Filtering (k=3 Features)

Thresholds found on **train set** (features 39, 111, 41).  
Ablation table and all evaluation metrics computed on **test set**.


In [13]:
# Activation distributions and threshold sweep — TRAIN SET
k3_acts_train = f_acts_train.numpy()
k3_acts_test  = f_acts_test.numpy()

y_bad_train = (metadata_train['rmsd'] > RMSD_THRESH).astype(int).values
y_bad_test  = (metadata_test['rmsd']  > RMSD_THRESH).astype(int).values
baseline_bad_train = y_bad_train.mean()
features = [39, 111, 41]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('k=3 features — activation distributions and threshold sweep (TRAIN set)', fontsize=12)

best_thrs = {}
for col, feat in enumerate(features):
    a      = k3_acts_train[:, feat]
    active = a > 0
    a_act  = a[active]
    y_act  = y_bad_train[active]

    print(f"\nFeature {feat} [train]: {active.sum():,} non-zero  "
          f"({active.mean()*100:.1f}%)  "
          f"{y_act.mean()*100:.1f}% bad (baseline {baseline_bad_train*100:.1f}%)")

    # Log-density histograms
    ax = axes[0, col]
    bins = np.linspace(0, a_act.max(), 60)
    ax.hist(a_act[y_act==0], bins=bins, density=True, alpha=0.6,
            color='mediumseagreen', label=f'good ({(y_act==0).sum():,})')
    ax.hist(a_act[y_act==1], bins=bins, density=True, alpha=0.6,
            color='salmon',         label=f'bad ({(y_act==1).sum():,})')
    ax.set_yscale('log')
    ax.set_xlabel('Activation value')
    ax.set_ylabel('Log density')
    ax.set_title(f'Feature {feat} — active samples (train)')
    ax.legend(fontsize=8)

    # Threshold sweep
    thresholds = np.linspace(a_act.min(), np.percentile(a_act, 99), 200)
    precisions, recalls, f1s = [], [], []
    valid_thrs = []
    for thr in thresholds:
        flagged = a >= thr
        if flagged.sum() == 0:
            continue
        prec = y_bad_train[flagged].mean()
        rec  = y_bad_train[flagged].sum() / y_bad_train.sum()
        f1   = 2*prec*rec / (prec+rec+1e-9)
        precisions.append(prec); recalls.append(rec); f1s.append(f1)
        valid_thrs.append(thr)

    best_idx = int(np.argmax(f1s))
    best_thr = valid_thrs[best_idx]
    best_thrs[feat] = best_thr

    ax2 = axes[1, col]
    ax2.plot(valid_thrs, precisions, color='steelblue',  lw=2, label='precision')
    ax2.plot(valid_thrs, recalls,    color='darkorange', lw=2, label='recall')
    ax2.plot(valid_thrs, f1s,        color='purple',     lw=2, label='F1')
    ax2.axhline(baseline_bad_train, color='grey', linestyle='--', lw=1,
                label=f'baseline {baseline_bad_train:.2f}')
    ax2.axvline(best_thr, color='red', linestyle=':', lw=1.5,
                label=f'best F1 thr={best_thr:.2f}')
    ax2.set_xlabel('Activation threshold')
    ax2.set_ylabel('Score')
    ax2.set_title(f'Feature {feat} — threshold sweep (train)')
    ax2.legend(fontsize=7)
    ax2.set_ylim(0, 1.05)

    print(f"  Best train-F1 threshold: {best_thr:.3f}  "
          f"prec={precisions[best_idx]:.3f}  rec={recalls[best_idx]:.3f}  "
          f"F1={f1s[best_idx]:.3f}  flagged={int((a>=best_thr).sum()):,}")

plt.tight_layout()
plt.show()

print("\nThresholds (found on train):", best_thrs)



Feature 39 [train]: 4,532 non-zero  (1.5%)  99.9% bad (baseline 87.4%)
  Best train-F1 threshold: 0.402  prec=0.999  rec=0.018  F1=0.035  flagged=4,532

Feature 111 [train]: 1,643 non-zero  (0.6%)  99.9% bad (baseline 87.4%)
  Best train-F1 threshold: 0.645  prec=0.999  rec=0.006  F1=0.013  flagged=1,643

Feature 41 [train]: 8,787 non-zero  (3.0%)  96.0% bad (baseline 87.4%)
  Best train-F1 threshold: 0.332  prec=0.960  rec=0.033  F1=0.063  flagged=8,787

Thresholds (found on train): {39: 0.4016227722167969, 111: 0.6452701091766357, 41: 0.33182287216186523}


In [22]:
# Apply train-derived thresholds to TEST SET
flags_test = {feat: (k3_acts_test[:, feat] >= best_thrs[feat]).astype(int)
              for feat in features}

baseline_bad_test = y_bad_test.mean()
print(f"Test baseline bad-pose rate: {baseline_bad_test:.3f}\n")
for feat in features:
    n_flag = flags_test[feat].sum()
    prec   = y_bad_test[flags_test[feat].astype(bool)].mean()
    rec    = y_bad_test[flags_test[feat].astype(bool)].sum() / y_bad_test.sum()
    f1     = 2*prec*rec/(prec+rec+1e-9)
    print(f"Feature {feat}: {n_flag:,} flagged  prec={prec:.3f}  rec={rec:.3f}  F1={f1:.3f}")


Test baseline bad-pose rate: 0.853

Feature 39: 243 flagged  prec=1.000  rec=0.004  F1=0.008
Feature 111: 24 flagged  prec=0.833  rec=0.000  F1=0.001
Feature 41: 0 flagged  prec=nan  rec=0.000  F1=nan


/var/folders/8n/lnh8wmjs2578l8skl0z15k3w0000gn/T/ipykernel_1760/2633949436.py:9: RuntimeWarning: Mean of empty slice.
  prec   = y_bad_test[flags_test[feat].astype(bool)].mean()
/opt/schrodinger/suites2025-3/internal/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [23]:
# Feature ablation — all 7 subsets, union rule, evaluated on TEST SET
records = []
for r in range(1, len(features) + 1):
    for combo in combinations(features, r):
        combined = np.zeros(len(y_bad_test), dtype=int)
        for feat in combo:
            combined |= flags_test[feat]
        flagged = combined.astype(bool)
        if flagged.sum() == 0:
            continue
        prec      = y_bad_test[flagged].mean()
        rec       = y_bad_test[flagged].sum() / y_bad_test.sum()
        f1        = 2*prec*rec / (prec+rec+1e-9)
        good_kept = (~flagged)[y_bad_test == 0].mean()
        records.append({
            'subset':      '+'.join(str(f) for f in combo),
            'n_flagged':   int(flagged.sum()),
            'pct_flagged': flagged.mean() * 100,
            'precision':   prec,
            'recall':      rec,
            'f1':          f1,
            'good_kept':   good_kept,
        })

ablation_df = pd.DataFrame(records)
print("Feature ablation — bad-pose filter (union rule) — TEST SET (2Å threshold)\n")
print(f"Baseline bad-pose rate: {baseline_bad_test:.3f}\n")
print(f"{'subset':<18} {'n_flag':>7} {'%flag':>6} {'prec':>7} {'recall':>7} {'F1':>7} {'good_kept':>10}")
print("-" * 68)
for _, row in ablation_df.iterrows():
    print(f"{row['subset']:<18} {row['n_flagged']:>7,} "
          f"{row['pct_flagged']:>6.1f}% "
          f"{row['precision']:>7.3f} {row['recall']:>7.3f} "
          f"{row['f1']:>7.3f} {row['good_kept']:>10.3f}")


Feature ablation — bad-pose filter (union rule) — TEST SET (2Å threshold)

Baseline bad-pose rate: 0.853

subset              n_flag  %flag    prec  recall      F1  good_kept
--------------------------------------------------------------------
39                     243    0.4%   1.000   0.004   0.008      1.000
111                     24    0.0%   0.833   0.000   0.001      1.000
39+111                 267    0.4%   0.985   0.004   0.009      1.000
39+41                  243    0.4%   1.000   0.004   0.008      1.000
111+41                  24    0.0%   0.833   0.000   0.001      1.000
39+111+41              267    0.4%   0.985   0.004   0.009      1.000


In [21]:
fig = plt.figure(figsize=(18, 10))
fig.suptitle('k=3 Feature Ablation — Bad-Pose Filter (thresholds from train, eval on test)', fontsize=13)

labels = ablation_df['subset'].tolist()
x      = np.arange(len(labels))
width  = 0.22

ax1 = fig.add_subplot(2, 3, (1, 2))
ax1.bar(x - 1.5*width, ablation_df['precision'],  width, label='Precision',  color='steelblue')
ax1.bar(x - 0.5*width, ablation_df['recall'],     width, label='Recall',     color='darkorange')
ax1.bar(x + 0.5*width, ablation_df['f1'],         width, label='F1',         color='purple')
ax1.bar(x + 1.5*width, ablation_df['good_kept'],  width, label='Good kept',  color='mediumseagreen')
ax1.axhline(baseline_bad_test, color='grey', linestyle='--', lw=1,
            label=f'bad baseline {baseline_bad_test:.2f}')
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
ax1.set_ylabel('Score')
ax1.set_title('Filter performance across feature subsets (test set)')
ax1.legend(fontsize=8, loc='upper left')
ax1.set_ylim(0, 1.1)

ax2 = fig.add_subplot(2, 3, 3)
bar_colors = ['#aec6e8']*3 + ['#f7c59f']*3 + ['#c5b0d5']
ax2.bar(x, ablation_df['n_flagged'], color=bar_colors)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
ax2.set_ylabel('N poses flagged (test)')
ax2.set_title('Flagged pose count per subset (test)')

# RMSD shift — test set
all_flagged = np.zeros(len(y_bad_test), dtype=int)
for feat in features:
    all_flagged |= flags_test[feat]
rmsd_clip = metadata_test['rmsd'].clip(upper=10).values
kept_mask = all_flagged == 0
bins = np.linspace(0, 10, 60)
ax3 = fig.add_subplot(2, 3, 4)
ax3.hist(rmsd_clip[kept_mask], bins=bins, density=True, alpha=0.5,
         color='mediumseagreen', label=f'after ({kept_mask.sum():,})')
ax3.hist(rmsd_clip,            bins=bins, density=True, alpha=0.4,
         color='salmon',        label=f'before ({len(rmsd_clip):,})')
ax3.axvline(RMSD_THRESH, color='black', linestyle='--', lw=1.5, label=f'{RMSD_THRESH}Å')
ax3.set_xlabel('RMSD (Å, clipped at 10)')
ax3.set_ylabel('Density')
ax3.set_title('RMSD distribution: before vs after\nall-3-feature union filter (test)')
ax3.legend(fontsize=8)

# Per-feature contribution breakdown — test
ax4 = fig.add_subplot(2, 3, 5)
others_test = {f: np.array([flags_test[g] for g in features if g != f]).any(axis=0)
               for f in features}
metrics = []
for feat in features:
    unique_bad  = (flags_test[feat]==1) & (~others_test[feat]) & (y_bad_test==1)
    shared_bad  = (flags_test[feat]==1) & (others_test[feat])  & (y_bad_test==1)
    false_flags = (flags_test[feat]==1) & (y_bad_test==0)
    metrics.append({'feat': str(feat),
                    'unique_bad': unique_bad.sum(),
                    'shared_bad': shared_bad.sum(),
                    'false_flags': false_flags.sum()})
mdf = pd.DataFrame(metrics)
bx  = np.arange(len(features))
ax4.bar(bx, mdf['unique_bad'],  label='Unique bad caught',      color='firebrick')
ax4.bar(bx, mdf['shared_bad'],  bottom=mdf['unique_bad'],
        label='Shared bad caught', color='salmon')
ax4.bar(bx, mdf['false_flags'],
        bottom=mdf['unique_bad']+mdf['shared_bad'],
        label='False flags (good poses)', color='lightgrey')
ax4.set_xticks(bx); ax4.set_xticklabels([f'feat {f}' for f in features])
ax4.set_ylabel('N poses (test)')
ax4.set_title('Per-feature flag breakdown (test)')
ax4.legend(fontsize=8)

# Co-flagging heatmap — test, bad poses
ax5 = fig.add_subplot(2, 3, 6)
bad_idx_te = y_bad_test == 1
flag_mat   = np.stack([flags_test[f][bad_idx_te] for f in features], axis=1).astype(float)
cov        = flag_mat.T @ flag_mat / bad_idx_te.sum()
im = ax5.imshow(cov, cmap='Blues', vmin=0, vmax=cov.max())
plt.colorbar(im, ax=ax5, label='co-flag rate (bad poses, test)')
ax5.set_xticks(range(len(features))); ax5.set_xticklabels([f'f{f}' for f in features])
ax5.set_yticks(range(len(features))); ax5.set_yticklabels([f'f{f}' for f in features])
for i in range(len(features)):
    for j in range(len(features)):
        ax5.text(j, i, f'{cov[i,j]:.2f}', ha='center', va='center', fontsize=10,
                 color='white' if cov[i,j] > 0.5*cov.max() else 'black')
ax5.set_title("Co-flagging rate (bad poses, test)\n(diagonal = per-feature recall)")

plt.tight_layout()
plt.show()


In [17]:
# Which test-set cases are flagged by each feature?
for feat in features:
    flagged_idx = np.where(flags_test[feat] == 1)[0]
    cases = metadata_test.iloc[flagged_idx]['case_id'].value_counts()
    print(f"\nFeature {feat} — {len(flagged_idx):,} test poses flagged across {len(cases)} cases")

union_mask = np.zeros(len(y_bad_test), dtype=bool)
for feat in features:
    union_mask |= flags_test[feat].astype(bool)

union_cases = metadata_test[union_mask].groupby('case_id').agg(
    n_flagged=('sample_idx', 'count'),
    n_bad_flagged=('rmsd', lambda x: (x > RMSD_THRESH).sum()),
    rmsd_min=('rmsd', 'min'),
    rmsd_mean=('rmsd', 'mean'),
).sort_values('n_flagged', ascending=False)

print(f"\nUnion filter (test) — {union_mask.sum():,} poses flagged across {len(union_cases)} cases")
print(union_cases.head(20).to_string())



Feature 39 — 243 test poses flagged across 2 cases

Feature 111 — 24 test poses flagged across 2 cases

Feature 41 — 0 test poses flagged across 0 cases

Union filter (test) — 267 poses flagged across 3 cases
                       n_flagged  n_bad_flagged  rmsd_min  rmsd_mean
case_id                                                             
throm_1ae8_throm_1d9i        244            244  3.824118   7.012029
ptp1b_1c84_ptp1b_2h4g         13              9  1.751735   3.587380
throm_1gj5_throm_1d3d         10             10  4.280205   6.743788


In [20]:
# Near-miss cases (test set): bad poses where the feature fires on some but not others
for feat in features:
    flagged = flags_test[feat].astype(bool)
    is_bad  = y_bad_test.astype(bool)

    near_miss_rows = []
    for case_id, group in metadata_test.groupby('case_id'):
        idx = group.index
        if (flagged[idx] & is_bad[idx]).any() and (~flagged[idx] & is_bad[idx]).any():
            fp = group[flagged[idx] & is_bad[idx]][['case_id', 'sample_idx', 'rmsd']].copy()
            mp = group[~flagged[idx] & is_bad[idx]][['case_id', 'sample_idx', 'rmsd']].copy()
            fp['status'] = 'flagged_bad'; mp['status'] = 'missed_bad'
            fp[f'feat{feat}_act'] = k3_acts_test[fp.index, feat]
            mp[f'feat{feat}_act'] = k3_acts_test[mp.index, feat]
            near_miss_rows.append(pd.concat([fp, mp]))

    if near_miss_rows:
        nm_df = pd.concat(near_miss_rows).reset_index(drop=True)
        out_path = f'../near_miss_feat{feat}_test.csv'
        nm_df.to_csv(out_path, index=False)
        print(f"Feature {feat}: {nm_df['case_id'].nunique()} near-miss cases → {out_path}")
    else:
        print(f"Feature {feat}: no near-miss cases found")


Feature 39: 2 near-miss cases → ../near_miss_feat39_test.csv
Feature 111: 2 near-miss cases → ../near_miss_feat111_test.csv
Feature 41: no near-miss cases found
